# Session 8: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over an alternative investments knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = ""

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = ""

## Task 2: Data Collection and Preparation

We'll be using the Alternative Investments Handbook - a comprehensive resource covering alternative investment strategies, reinsurance, private equity, real estate, commodities, and portfolio management.

### Data Preparation

We'll load the investments handbook as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/AlternativeInvestmentsHandbook.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
investment_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [4]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(investment_docs)}")
print(f"\nExample chunk:\n{investment_docs[0]}")

Raw documents: 1
Split chunks: 77

Example chunk:
page_content='The Alternative Investments Handbook
A Practical Guide to Non-Traditional Asset Classes and Risk Premiums

PART 1: FOUNDATIONS OF ALTERNATIVE INVESTING

Chapter 1: What Are Alternative Investments?' metadata={'source': 'data/AlternativeInvestmentsHandbook.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "investments_handbook".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    investment_docs,
    embeddings,
    location=":memory:",
    collection_name="investments_handbook",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

'The key due diligence steps for alternative investments include:\n\n1. **Assessing the Investment Strategy and Process:**\n   - Confirm that the strategy is clearly defined and repeatable.\n   - Understand the theoretical basis for how the strategy is expected to generate returns.\n\n2. **Evaluating the Manager and Track Record:**\n   - Review the manager’s performance across different market cycles.\n   - Determine if historical returns are consistent with the stated strategy.\n\n3. **Analyzing Risk Management:**\n   - Investigate the controls in place to limit losses.\n   - Understand how leverage is managed.\n   - Consider worst-case scenarios and risk mitigation systems.\n\n4. **Examining Liquidity and Redemption Terms:**\n   - Review lock-up periods, redemption notice requirements, and gate provisions.\n   - Assess whether these liquidity terms are appropriate given the investment strategy.\n\n5. **Understanding Fee Structures:**\n   - Evaluate management fees and performance fee

In [11]:
naive_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits primarily because its returns are largely uncorrelated with traditional financial markets like equities and bonds. This low correlation stems from the fact that reinsurance risks are driven by natural catastrophes such as hurricanes, earthquakes, and severe storms, rather than economic conditions. As a result, including reinsurance investments in a portfolio can reduce overall volatility and risk, especially during periods of financial market turmoil. Additionally, reinsurance offers attractive risk premiums and diversification across different geographies and peril types, further enhancing portfolio resilience and potential return profiles.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include:\n\n1. **Leveraged Buyouts (LBOs):** Acquiring mature companies using a combination of equity and debt financing. The aim is to improve operations, reduce costs, and grow revenue before exiting through a sale or IPO.\n\n2. **Growth Equity:** Investing in established companies that need capital to expand, often for geographic growth, product development, or acquisitions.\n\n3. **Distressed Investing:** Purchasing debt or equity of financially troubled companies at significant discounts, with value created through financial restructuring, operational improvements, or asset sales.\n\n4. **Secondaries:** Buying existing private equity fund interests from other investors, providing diversification and often at a discount to net asset value.\n\n5. **Event-Driven Strategies:** Investing around corporate events such as mergers, acquisitions, restructurings, or bankruptcies, including merger arbitrage.\n\nThese strategies encompass a ran

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(investment_docs)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

"The key due diligence steps for alternative investments include examining the following areas:\n\n1. Investment Strategy and Process: Ensure the strategy is clearly defined and repeatable, and understand the theoretical basis for expected returns.\n2. Track Record: Review the manager’s performance over multiple market cycles to assess consistency with the strategy.\n3. Risk Management: Evaluate the controls in place to limit losses, how leverage is managed, and analyze worst-case scenarios.\n4. Source of Return: Identify whether returns come from risk premiums, alpha, or structural advantages.\n5. Liquidity and Redemption Terms: Understand how liquid the investment is and the conditions for redemption.\n6. Fee Structure: Review the fees and how they align with performance.\n7. Manager’s Performance: Assess track record across different market environments.\n8. Risk Management Systems: Examine what risk controls and systems are implemented.\n9. Portfolio Fit: Consider how the investmen

In [16]:
bm25_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits primarily because it typically has a low correlation with traditional financial markets. This means that the returns from reinsurance investments, such as catastrophe bonds or collateralized reinsurance, often do not move in tandem with equities and bonds. During market downturns, especially those affecting traditional assets, reinsurance-related investments can maintain positive or neutral returns, helping to reduce overall portfolio volatility. For example, as noted in the context, alternative strategies like catastrophe bonds maintained positive or neutral returns during the 2008 financial crisis when equities suffered significant losses. This low correlation helps diversify the portfolio and can improve its risk-adjusted performance.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include investing in buyouts, venture capital, and growth equity. These strategies typically involve acquiring ownership stakes in private companies or startups, aiming for value creation and upside potential through operational improvements, strategic growth, or restructuring. Private equity investments are characterized by their illiquidity and longer investment horizons, with a focus on active management to enhance company performance before eventual exit through sales or initial public offerings.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

**Query:** "What are the specific correlation values between S&P 500 and commodities?"

**Why BM25 is better:**

1. **Exact numeric precision** - BM25 will precisely locate "Commodities: 0.1 to 0.3" from the correlation matrix reference, while embeddings might return semantically similar but less precise correlation information about other asset classes.

2. **Specific term matching** - BM25 excels at finding documents containing the exact phrase "S&P 500" and "commodities" together, ensuring you get the specific correlation data requested rather than general information about portfolio diversification or commodity investing benefits.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

'The key due diligence steps for alternative investments include:\n\n1. **Examining the Investment Strategy:** Understand the source of returns, such as risk premium, alpha, or structural advantages.\n\n2. **Evaluating the Manager:** Review the manager’s track record across various market conditions to assess their experience and effectiveness.\n\n3. **Assessing Operations and Legal Structure:** Ensure the operational processes are solid and the legal structure aligns with the investment’s objectives and compliance standards.\n\n4. **Analyzing Risk Management Systems:** Determine what risk management measures are in place to mitigate potential losses.\n\n5. **Liquidity and Redemption Terms:** Understand how liquid the investment is and the terms under which you can redeem your stake.\n\n6. **Fee Structure:** Review the fee structure carefully to see how fees are aligned with performance and their impact on returns.\n\n7. **Portfolio Fit:** Consider how the investment complements the ov

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits primarily because its returns are largely uncorrelated with traditional financial markets like equities and bonds. This is because the risks in reinsurance are driven by natural catastrophes and weather events rather than economic factors. Additionally, reinsurance risks can be spread across different geographies and peril types, further reducing correlation and helping diversify an investment portfolio.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include:\n\n1. Leveraged Buyouts (LBOs): Acquiring mature companies using a combination of equity and debt financing, with the goal of improving operations, reducing costs, and growing revenue before exiting through a sale or IPO.\n\n2. Event-Driven Strategies: Investing around corporate events such as mergers, acquisitions, restructurings, and bankruptcies. A common example is merger arbitrage, where investors buy the target company and short the acquirer in announced deals.\n\nThese strategies aim to generate returns by actively improving portfolio companies or capitalizing on corporate events.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!


In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

'The key due diligence steps for alternative investments include the following:\n\n1. **Examine the Investment Strategy and Process**: Determine if the strategy is clearly defined, repeatable, and understand the theoretical basis for expected returns. Assess whether the investment process aligns with the stated objectives.\n\n2. **Review the Manager’s Track Record**: Evaluate the manager’s performance across multiple market cycles, checking for consistency with the strategy and whether their performance is robust in different environments.\n\n3. **Assess Risk Management Systems**: Investigate the controls in place to limit losses, such as leverage management and risk controls. Understand how worst-case scenarios are handled.\n\n4. **Evaluate Liquidity and Redemption Terms**: Clarify lock-up periods, redemption notice requirements, gate provisions, and how these terms fit with the investment strategy.\n\n5. **Understand Fee Structure and Incentives**: Analyze management and performance 

In [26]:
multi_query_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits primarily because its returns are largely uncorrelated with traditional financial markets like equities and bonds. This is due to the fact that reinsurance risks are driven by natural catastrophes such as hurricanes, earthquakes, and severe storms, rather than economic factors. As a result, the performance of reinsurance investments tends to be independent of market fluctuations, helping to reduce overall portfolio volatility. Additionally, reinsurance and catastrophe-linked securities, such as catastrophe bonds, can offer attractive risk premiums and diversification across different geographies and peril types, further enhancing the risk-return profile of an investment portfolio.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include:\n\n1. Leveraged Buyouts (LBOs): Acquiring mature companies using a combination of equity and debt financing with the aim to improve operations, reduce costs, and grow revenue before exiting via sale or IPO.\n\n2. Venture Capital: Investing in early-stage or high-growth potential companies.\n\n3. Growth Equity: Infusing capital into established companies that need funding for expansion, such as geographic growth, new products, or acquisitions.\n\n4. Distressed Investing: Purchasing debt or equity of financially troubled companies at significant discounts, often aiming to generate value through restructuring or asset sales.\n\n5. Secondaries: Buying existing interests in private equity funds from other investors, offering diversification and potential discounts.\n\nThese strategies can involve various approaches and target different stages of company development or types of opportunities within the private equity universe.'

### Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

**How generating multiple reformulations improves recall:**

1. **Vocabulary mismatch mitigation** - Users and documents often use different terms for the same concepts. Multiple query reformulations increase the chances of matching the actual vocabulary used in relevant documents that might have been missed with a single query formulation.

2. **Semantic coverage expansion** - Different reformulations capture various aspects and angles of the same information need. For example, asking about "portfolio diversification," "risk reduction strategies," and "correlation benefits" might retrieve different but equally relevant documents about the same underlying concept, increasing the total relevant documents found.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [28]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [29]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="investments_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="investments_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [30]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [31]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [32]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [33]:
parent_document_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

"The key due diligence steps for alternative investments include:\n\n1. Evaluating the investment strategy and process:\n   - Is the strategy clearly defined and repeatable?\n   - What is the theoretical basis for generating returns?\n\n2. Assessing the manager's track record:\n   - How has the manager performed across multiple market cycles?\n   - Are returns consistent with the stated strategy?\n\n3. Reviewing risk management systems:\n   - What controls are in place to limit losses?\n   - How is leverage managed?\n   - What are the worst-case scenarios?\n\n4. Examining operational infrastructure:\n   - Is there adequate separation of duties between portfolio management and back-office operations?\n   - Are independent service providers (custodian, administrator, auditor) involved?\n\n5. Analyzing the fee structure:\n   - Are the fees reasonable relative to the value added?\n   - Is the fee structure aligned with investor interests (e.g., high-water marks, clawback provisions)?\n\n6.

In [34]:
parent_document_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits primarily because its returns are largely uncorrelated with traditional financial markets such as equity and bond markets. This is due to the fact that reinsurance risks are driven by natural catastrophic events like hurricanes, earthquakes, floods, and wildfires, which do not move in tandem with economic or financial market cycles. Additionally, reinsurance offers diversification across different geographies and peril types, further reducing overall portfolio risk. These characteristics enable investors to achieve risk spreading and potentially enhance portfolio stability by incorporating an asset class whose performance is driven by natural events rather than market fluctuations.\n'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include:\n\n1. **Leveraged Buyouts (LBOs):** Acquiring mature companies using a combination of equity and debt, with the goal of improving operations, reducing costs, and growing revenues before exiting via sale or IPO. Historically, LBOs have generated returns of 2-3 times the invested capital.\n\n2. **Growth Equity:** Investing in established companies that require capital to expand, such as geographic expansion, product development, or acquisitions. These companies are typically profitable.\n\n3. **Distressed Investing:** Purchasing the debt or equity of financially troubled companies at significant discounts, with value creation stemming from financial restructuring or operational improvements.\n\n4. **Secondaries:** Buying existing private equity fund interests from other investors, providing diversification across vintages and strategies, often at a discount to net asset value.\n\nThese strategies are outlined in the context provi

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [36]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [37]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [38]:
ensemble_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

'The key due diligence steps for alternative investments include:\n\n1. Examining the Investment Strategy and Process:\n   - Is the strategy clearly defined and repeatable?\n   - What is the theoretical basis for the expected returns?\n2. Assessing the Manager’s Track Record:\n   - Performance over multiple market cycles\n   - Consistency of returns with the stated strategy\n3. Evaluating Risk Management:\n   - Controls to limit losses\n   - Leverage management\n   - Worst-case scenario planning\n4. Reviewing Operational Infrastructure:\n   - Separation of duties between portfolio management and back-office functions\n   - Presence of independent service providers (custodian, administrator, auditor)\n5. Analyzing Fee Structures:\n   - Reasonableness relative to value added\n   - Alignment with investor interests (e.g., high-water marks, clawbacks)\n6. Understanding Liquidity Terms:\n   - Lock-up periods, redemption notice requirements\n   - Gate provisions and alignment with the invest

In [39]:
ensemble_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits primarily because its returns are largely uncorrelated with traditional financial markets like equities and bonds. This low correlation arises because reinsurance investments are driven by natural catastrophes and weather events—such as hurricanes, earthquakes, and storms—rather than economic conditions or financial market movements. Additionally, reinsurance offers diversifiable risk across different geographies and peril types, further reducing overall portfolio volatility. By including reinsurance in an investment portfolio, investors can potentially lower risk and improve risk-adjusted returns through exposure to these non-market-linked risks.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include:\n\n1. Leveraged Buyouts (LBOs): Acquiring mature companies using a combination of equity and debt, aiming to improve operations, reduce costs, and grow revenue before exiting through a sale or IPO.\n\n2. Growth Equity: Investing in established, profitable companies that require capital to expand, such as for geographic growth, product development, or acquisitions.\n\n3. Distressed Investing: Purchasing debt or equity of financially troubled companies at significant discounts, with value creation through restructuring, operational improvement, or asset sales.\n\n4. Secondaries: Buying existing private equity fund interests from other investors, providing diversification across vintage years and strategies, often at a discount to net asset value.\n\n5. Event-Driven Strategies: Investing around corporate events such as mergers, acquisitions, restructurings, and bankruptcies, including merger arbitrage.\n\nEach of these strategies 

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [41]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [42]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [43]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="investments_handbook_semantic_chunks"
)

We'll use naive retrieval for this example.

In [44]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [45]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [46]:
semantic_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

'The key due diligence steps for alternative investments include examining the following areas:\n\n1. Investment Strategy and Process:\n   - Ensure the strategy is clearly defined and repeatable.\n   - Understand the theoretical basis for generating returns.\n\n2. Track Record:\n   - Review the manager’s performance over multiple market cycles.\n   - Verify that returns are consistent with the stated strategy.\n\n3. Risk Management:\n   - Assess the controls in place to limit losses.\n   - Evaluate how leverage is managed.\n   - Consider worst-case scenario planning.\n\n4. Operational Infrastructure:\n   - Confirm there is adequate separation of duties between portfolio management and back-office operations.\n   - Check for independent service providers for custody, administration, and auditing.\n\n5. Fee Structure:\n   - Analyze whether fees are reasonable relative to value added.\n   - Ensure fee structures align with investor interests via high-water marks and clawback provisions.\n

In [47]:
semantic_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits because its returns are largely uncorrelated with traditional financial markets such as equities and bonds. This is because reinsurance risks are driven by natural catastrophes, weather events, and other environmental factors rather than economic or market pressures. For investors, this means that reinsurance investments can help reduce overall portfolio volatility and risk, especially during periods of market stress, since their performance does not typically move in tandem with stock or bond markets. Additionally, reinsurance often offers attractive risk premiums and short-duration income streams, further enhancing diversification and potential returns when included as part of a broader investment portfolio.'

In [48]:
semantic_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include:\n\n1. Leveraged Buyouts (LBOs): Acquiring mature companies using a combination of equity and debt financing, with the goal of improving operations, reducing costs, and growing revenue before exiting through a sale or IPO.\n\n2. Growth Equity: Investing in established companies that require capital to expand their operations or market presence.\n\n3. Venture Capital: Funding early-stage, high-growth potential companies, particularly in technology and healthcare sectors, across stages such as seed, Series A, and later rounds.\n\n4. Distressed Investing: Purchasing debt or equity of financially troubled companies at significant discounts, with value creation through restructuring or asset sales.\n\n5. Secondaries: Buying existing private equity fund interests from other investors, offering diversification and often at a discount to net asset value.\n\nThese strategies focus on different types of companies, stages of development, a

### Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

**How semantic chunking might behave with short, repetitive sentences:**

1. **Over-fragmentation** - Short sentences with high semantic similarity would likely be grouped together excessively, creating very small chunks that lack sufficient context for effective retrieval.

2. **Loss of question-answer pairing** - FAQ structure might be broken apart, separating questions from their corresponding answers if they have different semantic content.

**Algorithm adjustments:**

1. **Use structure-aware chunking** - Implement custom logic to keep Q&A pairs together regardless of semantic similarity, treating each FAQ pair as a minimum chunk unit.

2. **Adjust threshold parameters** - Use a more restrictive threshold (e.g., higher percentile or standard deviation) to prevent over-grouping of similar but distinct FAQ entries, ensuring each maintains its unique context.

---

# Breakout Room Part #2

### Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.